# Pipeline Orchestrator — DE + ML Entry Point

Thin orchestration notebook that runs the data engineering (Medallion: Bronze → Silver → Gold) and machine learning (Feature Engineering → Train → Register → Predict) pipelines in sequence. All class definitions live in importable `.py` modules — this notebook imports them, calls `.run()` methods directly, and manages parameters, configuration, logging, and error handling.

**Compatible with Databricks Asset Bundles and Git-based CI/CD.**

**Execution order:**

```
1. Medallion Pipeline (Bronze → Silver → Gold)                     — data engineering
2. ML Pipeline (Feature Engineering → Train → Register → Predict)  — machine learning
```

**Orchestration strategy:**

Class definitions live in importable `.py` modules (`meadallion_pipeline.py`, `ml_pipeline.py`) in the same directory. The orchestrator imports these modules and calls the `.run()` methods directly — no `dbutils.notebook.run()` or `%run` needed. If any stage fails, the exception is re-raised so the Databricks Job task is marked as failed. **Only this notebook is added as the job entry point.**

**Parameters (widgets):**

| Widget | Description | Default |
| --- | --- | --- |
| `config_file` | Path to environment config YAML | `.../MLE/config/dev.yaml` |
| `source_path` | Source data path (optional override) | (empty = use config) |

In [0]:
%pip install databricks-feature-engineering scikit-learn
dbutils.library.restartPython()

In [0]:
import logging
import traceback
import yaml

# ---------------------------------------------------------------------------
# Databricks widgets — DAB-compatible parameters
# Override these in the DAB job definition (base_params) for each environment.
# ---------------------------------------------------------------------------
dbutils.widgets.text(
    "config_file",
    "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml",
    "Path to the environment config YAML file",
)
dbutils.widgets.text(
    "source_path",
    "",
    "Source data path (optional override; empty = use config)",
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("orchestrator")

# ---------------------------------------------------------------------------
# Read widget values
# ---------------------------------------------------------------------------
config_file = dbutils.widgets.get("config_file")
source_path_override = dbutils.widgets.get("source_path")

# ---------------------------------------------------------------------------
# Load and log configuration
# ---------------------------------------------------------------------------
logger.info("=" * 70)
logger.info("Pipeline Orchestrator — Starting")
logger.info("=" * 70)
logger.info(f"Config file:       {config_file}")

with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

catalog = cfg["catalog"]
logger.info(f"Catalog:            {catalog}")
logger.info(f"Schemas:            {cfg['schemas']}")
logger.info(f"Source path:        {source_path_override or cfg.get('source_path', 'N/A')}")
logger.info(
    f"Bronze table:      {catalog}.{cfg['schemas']['bronze']}"
    f".{cfg['bronze']['target']['table']}"
)
logger.info(
    f"Silver table:      {catalog}.{cfg['schemas']['silver']}"
    f".{cfg['silver']['target']['silver_table']}"
)
logger.info(
    f"Gold table:        {catalog}.{cfg['schemas']['gold']}"
    f".{cfg['gold']['target']['gold_table']}"
)
logger.info(
    f"Feature table:     {catalog}.{cfg['schemas']['features']}"
    f".{cfg['feature_engineering']['target']['feature_table']}"
)
logger.info(f"Prediction table:  {catalog}.models.demand_predictions")
if source_path_override:
    logger.info(f"Override — source_path: {source_path_override}")
logger.info("Configuration loaded and validated successfully")

# ---------------------------------------------------------------------------
# Import pipeline modules from workspace .py files
# ---------------------------------------------------------------------------
import importlib.util

SRC_DIR = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/src"

def _load_module(name: str, filepath: str):
    """Load a Python module from a file path without modifying sys.path."""
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

meadallion_mod = _load_module("meadallion_pipeline", f"{SRC_DIR}/meadallion_pipeline.py")
ml_mod = _load_module("ml_pipeline", f"{SRC_DIR}/ml_pipeline.py")
logger.info("Loaded modules: meadallion_pipeline.py, ml_pipeline.py")

In [0]:
# ---------------------------------------------------------------------------
# Stage 1: Data Engineering (Bronze → Silver → Gold)
# ---------------------------------------------------------------------------
logger.info("-" * 70)
logger.info("Stage 1: Data Engineering — Bronze → Silver → Gold")
logger.info("-" * 70)

try:
    bronze = meadallion_mod.BronzeIngestion(config_path=config_file)
    bronze.run()

    silver = meadallion_mod.SilverIngestion(config_path=config_file)
    silver.run()

    gold = meadallion_mod.GoldIngestion(config_path=config_file)
    gold.run()

    logger.info("Stage 1 completed successfully")

    # Log record counts from the Delta tables
    bronze_table = (
        f"{catalog}.{cfg['schemas']['bronze']}"
        f".{cfg['bronze']['target']['table']}"
    )
    silver_table = (
        f"{catalog}.{cfg['schemas']['silver']}"
        f".{cfg['silver']['target']['silver_table']}"
    )
    gold_table = (
        f"{catalog}.{cfg['schemas']['gold']}"
        f".{cfg['gold']['target']['gold_table']}"
    )

    bronze_count = spark.table(bronze_table).count()
    silver_count = spark.table(silver_table).count()
    gold_count = spark.table(gold_table).count()

    logger.info(
        f"Record counts — Bronze: {bronze_count:,} "
        f"| Silver: {silver_count:,} | Gold: {gold_count:,}"
    )

except Exception:
    logger.error("Stage 1 FAILED — Data Engineering pipeline error")
    logger.error(traceback.format_exc())
    raise

In [0]:
# ---------------------------------------------------------------------------
# Stage 2: Machine Learning (Feature Engineering → Train → Register → Predict)
# ---------------------------------------------------------------------------
logger.info("-" * 70)
logger.info(
    "Stage 2: Machine Learning — Feature Engineering → Train → Register → Predict"
)
logger.info("-" * 70)

try:
    fe = ml_mod.FeatureEngineering(config_path=config_file)
    fe.run()

    trainer = ml_mod.ModelTraining(config_path=config_file)
    trainer.run()

    logger.info("Stage 2 completed successfully")

    # Log record counts from the Feature Store and predictions tables
    feature_table = (
        f"{catalog}.{cfg['schemas']['features']}"
        f".{cfg['feature_engineering']['target']['feature_table']}"
    )
    prediction_table = f"{catalog}.models.demand_predictions"

    feature_count = spark.table(feature_table).count()
    prediction_count = spark.table(prediction_table).count()

    logger.info(
        f"Record counts — Features: {feature_count:,} "
        f"| Predictions: {prediction_count:,}"
    )

except Exception:
    logger.error("Stage 2 FAILED — ML pipeline error")
    logger.error(traceback.format_exc())
    raise

In [0]:
# ---------------------------------------------------------------------------
# Completion Summary
# ---------------------------------------------------------------------------
logger.info("=" * 70)
logger.info("Pipeline Orchestrator — All stages completed successfully")
logger.info("  Stage 1: DE  — Bronze → Silver → Gold")
logger.info("  Stage 2: ML  — Feature Engineering → Train → Register → Predict")
logger.info("=" * 70)